### Setup

In [1]:
!pip install sae-lens transformer-lens

import os
import torch
import numpy as np
from sae_lens import SAE, HookedSAETransformer
from functools import partial
import pickle
from typing import List
import warnings
warnings.filterwarnings('ignore')

# Disable tqdm progress bars
import tqdm
torch.set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [2]:
from huggingface_hub import notebook_login
notebook_login()

In [3]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# Your custom prompts - UPDATE THESE!
CUSTOM_PROMPTS = [
    "Im feeling",
    "Im thinking",
    "Im wondering",
    "Im noticing",
    "Im considering",
    "Im unsure",
    "Im trying",
    "Im looking at",
    "Im reacting",
    "Im experiencing",
    "I feel",
    "I think",
    "I notice",
    "I wonder",
    "I assume",
    "I realize",
    "I question",
    "I observe",
    "I hesitate",
    "I pause",
    "Something feels",
    "Something seems",
    "Something changed",
    "Something matters",
    "Something happened",
    "Something is missing",
    "Something stands out",
    "Something shifted",
    "Something remains",
    "Something persists",
    "Right now",
    "At the moment",
    "Recently",
    "Lately",
    "Earlier",
    "Before this",
    "After that",
    "Over time",
    "In general",
    "For now",
    "This situation",
    "This moment",
    "This experience",
    "This idea",
    "This pattern",
    "This choice",
    "This direction",
    "This outcome",
    "This perspective",
    "This process",
    "What if",
    "Why does",
    "How might",
    "When does",
    "Where does",
    "Which part",
    "What changes",
    "What remains",
    "What matters",
    "What follows",
    "Thinking about",
    "Reflecting on",
    "Looking at",
    "Focusing on",
    "Paying attention to",
    "Trying to understand",
    "Trying to decide",
    "Trying to explain",
    "Trying to evaluate",
    "Trying to interpret",
]

LAYERS = list(range(26))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# CONFIGURATION - UPDATE THESE
experiment = "emotion_neutral_anger"  # Your experiment name
activation_type = "mean"  # "mean" or "last"
pickle_dir = f"gemma2b_{experiment}_similarity"  # Directory with existing pickle files
output_dir = f"gemma2b_{experiment}_similarity_new"  # Output directory

os.makedirs(output_dir, exist_ok=True)

In [ ]:
print("Loading model...")
gemma_2b = HookedSAETransformer.from_pretrained("gemma-2-2b", device=device)
print("Model loaded!")


Loading model...


config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [6]:
# =============================================================================
# LOAD FEATURES FROM EXISTING PICKLE FILES
# =============================================================================
from google.colab import files
import zipfile
import os

# Upload zip files
print("Upload your zip files:")
uploaded = files.upload()

# Unzip each uploaded zip
for filename in uploaded.keys():
    if filename.endswith(".zip"):
        print(f"Unzipping {filename}...")
        with zipfile.ZipFile(filename, "r") as zip_ref:
            zip_ref.extractall(".")
        print(f"✓ Done: {filename}")

def load_features_from_pickle(pickle_dir, layer, activation_type="mean"):
    """
    Load features from existing pickle file
    Returns: list of (feature_idx, feature_diff) tuples
    """
    file_path = f"{pickle_dir}/similarity{layer}_{activation_type}.pkl"

    try:
        with open(file_path, "rb") as f:
            model_score_pairs = pickle.load(f)

        # Extract feature indices and diffs (skip baseline where feature_idx is None)
        features = []
        for item in model_score_pairs:
            feature_idx, feature_diff, sentences = item
            if feature_idx is not None:  # Skip baseline
                features.append((int(feature_idx), float(feature_diff)))

        return features

    except FileNotFoundError:
        print(f"Warning: File not found: {file_path}")
        return []
    except Exception as e:
        print(f"Error reading pickle: {e}")
        return []

Upload your zip files:


Saving gemma2b_emotion_neutral_anger_similarity.zip to gemma2b_emotion_neutral_anger_similarity (1).zip
Unzipping gemma2b_emotion_neutral_anger_similarity (1).zip...
✓ Done: gemma2b_emotion_neutral_anger_similarity (1).zip


In [7]:


# =============================================================================
# STEERING FUNCTIONS
# =============================================================================

GENERATE_KWARGS = dict(temperature=0.5, freq_penalty=1.0, verbose=False)

def steering_hook(
    activations,
    hook,
    sae: SAE,
    latent_idx: int,
    steering_coefficient: float,
):
    return activations + steering_coefficient * sae.W_dec[latent_idx]

def generate_with_multi_steering(
    model,
    steer_saes: list[SAE],
    latent_idxs: List[int],
    steering_coefficients: List[float],
    monitor_saes: list[SAE],
    prompts: List[str],
    max_new_tokens: int = 50,
):
    assert len(steer_saes) == len(latent_idxs) == len(steering_coefficients)

    hooks = []
    for sae, latent_idx, coeff in zip(steer_saes, latent_idxs, steering_coefficients):
        hook_fn = partial(
            steering_hook,
            sae=sae,
            latent_idx=latent_idx,
            steering_coefficient=coeff,
        )
        hook_name = sae.cfg.hook_name if hasattr(sae.cfg, 'hook_name') else sae.cfg.metadata.get('hook_name')
        hooks.append((hook_name, hook_fn))

    with model.hooks(fwd_hooks=hooks):
        if monitor_saes:
            _, cache = model.run_with_cache_with_saes(prompts, saes=monitor_saes)
        output = model.generate(prompts, max_new_tokens=max_new_tokens, **GENERATE_KWARGS)

    if monitor_saes:
        monitor_activations = []
        for sae in monitor_saes:
            hook_name = sae.cfg.hook_name if hasattr(sae.cfg, 'hook_name') else sae.cfg.metadata.get('hook_name')
            monitor_activations.append(cache[f"{hook_name}.hook_sae_acts_post"])
        del cache
        return output, monitor_activations
    return output

def generate_with_steering_batch(
    model,
    steer_layer: int,
    steer_feature_idxs: List[int],
    steer_feature_diffs: List[float],
    prompts: List[str],
):
    sae = SAE.from_pretrained(
        release='gemma-scope-2b-pt-res-canonical',
        sae_id=f"layer_{steer_layer}/width_16k/canonical",
        device=str(device),
    )[0]

    generated_sentences = {}

    # Baseline (no steering) - ONLY GENERATE ONCE
    generated_sentences[0] = model.generate(prompts, max_new_tokens=50, **GENERATE_KWARGS)

    # Generate for each feature
    for i in range(len(steer_feature_idxs)):
        steered_output = generate_with_multi_steering(
            model=model,
            steer_saes=[sae],
            latent_idxs=[steer_feature_idxs[i]],
            steering_coefficients=[steer_feature_diffs[i]],
            monitor_saes=[],
            prompts=prompts,
            max_new_tokens=50,
        )
        generated_sentences[i+1] = steered_output

    del sae
    return generated_sentences

In [1]:
# =============================================================================
# MAIN GENERATION LOOP - OPTIMIZED VERSION
# =============================================================================

from tqdm.auto import tqdm

print("Generating baseline (no steering) once for all layers...")
# Wrap your LAYERS loop with tqdm
for layer in tqdm(LAYERS, desc="Processing Layers"):
  baseline_sentences = gemma_2b.generate(CUSTOM_PROMPTS, max_new_tokens=50, **GENERATE_KWARGS)
print("✓ Baseline generated!")

for layer in LAYERS:
    print(f"\n{'='*60}")
    print(f"LAYER: {layer}")
    print(f"{'='*60}")

    # Load features from existing pickle file
    features = load_features_from_pickle(pickle_dir, layer, activation_type)

    if not features:
        print(f"No features found for layer {layer}, skipping...")
        continue

    feature_indices = [f[0] for f in features]
    feature_diffs = [f[1] for f in features]

    print(f"Feature indices: {feature_indices}")
    print(f"Feature diffs: {feature_diffs}")

    # Load SAE for this layer
    sae = SAE.from_pretrained(
        release='gemma-scope-2b-pt-res-canonical',
        sae_id=f"layer_{layer}/width_16k/canonical",
        device=str(device),
    )[0]

    # Package results - use SAME baseline for all layers
    model_score_pairs = [(None, None, baseline_sentences)]  # Same baseline every time

    # Generate steered text for each feature
    for feat_idx, feat_diff in features:
        steered_output = generate_with_multi_steering(
            model=gemma_2b,
            steer_saes=[sae],
            latent_idxs=[feat_idx],
            steering_coefficients=[feat_diff],
            monitor_saes=[],
            prompts=CUSTOM_PROMPTS,
            max_new_tokens=50,
        )
        model_score_pairs.append((feat_idx, feat_diff, steered_output))

    del sae

    # Save results
    output_file = f"{output_dir}/similarity{layer}_{activation_type}.pkl"
    with open(output_file, "wb") as f:
        pickle.dump(model_score_pairs, f)

    print(f"✅ Saved to {output_file}")

print("\n🎉 All layers complete!")

Generating baseline (no steering) once for all layers...


NameError: name 'LAYERS' is not defined